# F1 Strategist — GRPO Training & Evaluation

**Hackathon:** Meta PyTorch OpenEnv Grand Finale · April 25–26 2026 · Bangalore  
**Team:** Shashwat Rajan · Tanish Shitanshu · Org: [Deltasthic](https://huggingface.co/Deltasthic)  
**Space:** [Deltasthic/f1-strategist](https://huggingface.co/spaces/Deltasthic/f1-strategist)  
**GitHub:** [Deltasthicc/F1_Simulator_OpenENV](https://github.com/Deltasthicc/F1_Simulator_OpenENV)

---

## What this notebook does

| Section | What it does | GPU needed? |
|---------|-------------|-------------|
| 1 | Install deps + clone repo | No |
| 2 | Environment smoke test (reset → step → score) | No |
| 3 | Evaluation results (4 policies × 4 scenarios) | No |
| 4 | Training curve — GRPO reward over 800 steps | No |
| 5 | Before / After race story | No |
| 6 | Track generalization grid (8 circuits) | No |
| 7 | *(Optional)* Run GRPO training yourself | Yes — T4/A100 |

Sections 1–6 run on Colab CPU (no GPU needed). Section 7 requires a T4 or better.

---

## Environment summary

The LLM acts as the **race engineer on the pit wall**. Each step is one lap. The model sees the current race state (lap, position, tyre health, fuel, weather, opponent gaps) and issues one strategic command from a 21-command vocabulary.

**Six-dimensional reward (summed to 0–1 range):**
- Race result (×0.30): final championship position  
- Strategic decisions (×0.25): correct pit calls, safety-car windows, undercut timing  
- Tyre management (×0.20): staying within the cliff, preserving health  
- Fuel management (×0.10): not over- or under-fuelling  
- Comms quality (×0.10): appropriate radio messages before key decisions  
- Operational efficiency (×0.05): avoiding redundant or harmful actions  

**Long-horizon design:** lap-4 decisions (pit timing, tyre choice) manifest in lap-9 outcomes (rivals catching up, cliff degradation). This is exactly Theme #2 of the hackathon — Super Long-Horizon Planning.

## Section 1 — Install dependencies and clone the repo

In [ ]:
# Install runtime dependencies
%pip install -q "openenv-core>=0.2.3" fastapi uvicorn "pydantic>=2.0" \
                numpy pandas matplotlib pillow imageio tqdm

import pathlib, subprocess, sys, os

REPO_URL  = "https://github.com/Deltasthicc/F1_Simulator_OpenENV.git"
REPO_NAME = "F1_Simulator_OpenENV"

# Clone once; pull if the directory already exists
if not pathlib.Path(REPO_NAME).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    print(f"Cloned into {REPO_NAME}/")
else:
    subprocess.run(["git", "-C", REPO_NAME, "pull", "--ff-only"], check=True)
    print("Repo already present — pulled latest.")

# Add project root to sys.path so all modules resolve correctly
ROOT = pathlib.Path(REPO_NAME).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print(f"Working directory: {ROOT}")

## Section 2 — Environment smoke test

Verify `reset()`, `step()`, and the reward signal work end-to-end without a server.

In [ ]:
from server.environment import F1StrategistEnvironment
from models import F1Action

env  = F1StrategistEnvironment()
obs  = env.reset(task="weather_roulette", seed=7)

print(f"Episode reset — task=weather_roulette  seed=7")
print(f"  Total laps : {obs.total_laps}")
print(f"  Start pos  : P{obs.ego_position}")
print(f"  Compound   : {obs.ego_tyre_compound}")
print(f"  Fuel       : {obs.ego_fuel_remaining_kg:.1f} kg")
print()

# Run 5 laps with a simple script
DEMO_ACTIONS = [
    "REQUEST_FORECAST",
    "STAY_OUT",
    "STAY_OUT",
    'RADIO_DRIVER "Box now for inters — rain coming."',
    "PIT_NOW inter",
]

print(f"{'Lap':>4}  {'Pos':>3}  {'Compound':>8}  {'Health':>7}  {'Fuel':>6}  Action")
print("-" * 72)
for act in DEMO_ACTIONS:
    print(f"{obs.current_lap:>4}  P{obs.ego_position:<2}  {obs.ego_tyre_compound:>8}"
          f"  {obs.ego_tyre_health_pct:>6.1f}%  {obs.ego_fuel_remaining_kg:>5.1f}kg  {act}")
    obs = env.step(F1Action(command=act))

print(f"\nAfter 5 laps: P{obs.ego_position}  compound={obs.ego_tyre_compound}  "
      f"health={obs.ego_tyre_health_pct:.1f}%  score={obs.score:.3f}")
print("\nSmoke test PASSED.")

In [ ]:
# Run a full episode with the heuristic policy and print the lap log
from inference import run_inference

scores = run_inference(
    model="heuristic",
    task="weather_roulette",
    n_episodes=1,
    verbose=True,
    seed=7,
)
print(f"\nFinal score: {scores[0]:.3f}")

## Section 3 — Evaluation results: 4 policies × 4 scenarios

Results from the full evaluation run on the SSH server (RTX 5090). The trained model is `grpo_v1` — Qwen3-4B + LoRA fine-tuned with 800 GRPO steps.

In [ ]:
import json, pathlib

summary_path = pathlib.Path("results/eval_summary.json")
with summary_path.open() as f:
    summary = json.load(f)

policies  = ["random", "untrained", "trained", "expert"]
scenarios = list(next(iter(summary.values())).keys())

# Pretty-print the table
col_w = 20
header = f"{'Scenario':<28}" + "".join(f"{p.upper():>{col_w}}" for p in policies)
print(header)
print("-" * len(header))
for sc in scenarios:
    row = f"{sc:<28}"
    for p in policies:
        v = summary.get(p, {}).get(sc, {}).get("mean", 0.0)
        row += f"{v:>{col_w}.3f}"
    print(row)

print()
# Delta: trained vs untrained
print("Delta (trained − untrained):")
for sc in scenarios:
    tr = summary.get("trained", {}).get(sc, {}).get("mean", 0.0)
    un = summary.get("untrained", {}).get(sc, {}).get("mean", 0.0)
    print(f"  {sc:<28} +{tr - un:+.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PALETTE = {
    "random":    "#444444",
    "untrained": "#e97c1a",
    "trained":   "#00d2be",
    "expert":    "#e10600",
}
SC_LABELS = {
    "dry_strategy_sprint":  "Dry Sprint",
    "weather_roulette":     "Weather Roulette",
    "late_safety_car":      "Late Safety Car",
    "championship_decider": "Championship Decider",
}

x   = np.arange(len(scenarios))
w   = 0.18
fig, ax = plt.subplots(figsize=(12, 5.5), facecolor="#0c0c0c")
ax.set_facecolor("#0c0c0c")

for i, pol in enumerate(policies):
    vals = [summary.get(pol, {}).get(sc, {}).get("mean", 0.0) for sc in scenarios]
    bars = ax.bar(x + (i - 1.5) * w, vals, w,
                  label=pol, color=PALETTE[pol], alpha=0.9, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                f"{v:.2f}", ha="center", va="bottom", fontsize=7,
                color="#dddddd", fontfamily="monospace")

# Delta arrows for trained vs untrained
for i, sc in enumerate(scenarios):
    tr = summary.get("trained", {}).get(sc, {}).get("mean", 0.0)
    un = summary.get("untrained", {}).get(sc, {}).get("mean", 0.0)
    ax.annotate("", xy=(i + 0.5 * w, tr + 0.04), xytext=(i - 0.5 * w, un + 0.04),
                arrowprops=dict(arrowstyle="->", color="#00d2be", lw=1.2))

ax.set_xticks(x)
ax.set_xticklabels([SC_LABELS.get(sc, sc) for sc in scenarios],
                   color="#cccccc", fontsize=9)
ax.set_ylim(0, 1.12)
ax.axhline(0.50, color="#ffffff", ls="--", lw=0.6, alpha=0.35, zorder=1, label="0.50 baseline")
ax.set_ylabel("Mean score (0–1)", color="#999", fontsize=9)
ax.set_title("F1 Strategist — Policy Comparison (4 scenarios)",
             color="#eeeeee", fontsize=12, pad=14)
ax.tick_params(colors="#666")
for spine in ax.spines.values():
    spine.set_edgecolor("#222")
ax.yaxis.grid(True, color="#1e1e1e", zorder=0)
leg = ax.legend(fontsize=8, framealpha=0.15, labelcolor="white",
                loc="upper right", facecolor="#111")
plt.tight_layout()
plt.savefig("results/eval_curve.png", dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Saved to results/eval_curve.png")

## Section 4 — GRPO training curve

The reward curve from the actual training run (800 steps, Qwen3-4B + LoRA, RTX 5090). The trained checkpoint is `grpo_v1` in the repo root.

In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

curve_path = pathlib.Path("results/training_loss_curve.png")
if curve_path.exists():
    display(IPyImage(str(curve_path), width=900))
    print(f"Training curve loaded from {curve_path}")
else:
    print("Training curve not found. Run Section 7 to generate it.")

## Section 5 — Before / After race story

Scenario: **Weather Roulette, Spa, seed 7.** The untrained model (left) stays on dry tyres into heavy rain, losing 4 positions. The GRPO-trained model (right) correctly calls the pit for inters on lap 6, maintains P2, and scores 0.935 vs 0.378.

In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

story_path = pathlib.Path("results/race_story.png")
if story_path.exists():
    display(IPyImage(str(story_path), width=980))
    print(f"Race story loaded from {story_path}")
else:
    # Regenerate if missing
    import subprocess
    result = subprocess.run(
        ["python", "scripts/plot_race_story.py",
         "--task", "weather_roulette", "--seed", "7",
         "--out", "results/race_story.png"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        display(IPyImage("results/race_story.png", width=980))
    else:
        print("Could not generate race story:", result.stderr[-400:])

## Section 6 — Track generalization grid

The expert policy evaluated across 8 F1 circuits, each paired with its most relevant scenario family. This shows the environment generalizes across full circuit layouts, not just a single track.

In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

grid_path = pathlib.Path("results/track_grid.png")
if grid_path.exists():
    display(IPyImage(str(grid_path), width=980))
    print(f"Track grid loaded from {grid_path}")
else:
    import subprocess
    result = subprocess.run(
        ["python", "scripts/render_track_grid.py", "--out", "results/track_grid.png"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        display(IPyImage("results/track_grid.png", width=980))
    else:
        print("Could not generate track grid:", result.stderr[-400:])

## Section 7 — Run GRPO training yourself *(optional, GPU required)*

This section reproduces the training run. You need a T4 or better. Expected runtime:
- T4 (16 GB): ~90 minutes for 400 steps  
- A100 (40 GB): ~35 minutes for 800 steps  

The training script uses TRL's `GRPOTrainer` with Unsloth acceleration.

In [ ]:
# Check GPU availability
import subprocess, sys
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("No GPU detected. Sections 1–6 work on CPU; Section 7 needs a GPU runtime.")
        print("In Colab: Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("torch not installed yet")

In [ ]:
# Install training dependencies (only needed for Section 7)
%pip install -q \
    "trl==0.14.0" \
    "transformers>=4.55.4" \
    "peft>=0.13.2" \
    "datasets>=3.2.0" \
    "accelerate>=1.3" \
    "bitsandbytes>=0.49" \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("Training deps installed.")

In [ ]:
# Generate the SFT warm-start dataset from expert trajectories
# This step is fast (~30 seconds on CPU)
import subprocess, pathlib

sft_path = pathlib.Path("data/sft_dataset.jsonl")
if not sft_path.exists():
    result = subprocess.run(
        ["python", "baselines/expert_solver.py", "--n-episodes", "50",
         "--out", str(sft_path)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"SFT dataset created: {sft_path} ({sft_path.stat().st_size // 1024} KB)")
    else:
        print("expert_solver output:", result.stdout[-400:])
        print(result.stderr[-400:])
else:
    print(f"SFT dataset already exists: {sft_path}")

In [ ]:
# Smoke-run GRPO training for 5 steps — verifies the pipeline without a long wait
# Remove --max-steps 5 and increase batch/steps for a full run
import subprocess

result = subprocess.run(
    [
        "python", "train.py",
        "--backend", "trl",
        "--max-steps", "5",
        "--batch-size", "1",
        "--grad-accum", "4",
        "--output-dir", "./grpo_smoke",
        "--task", "weather_roulette",
    ],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Full GRPO training run — 800 steps on all scenario families
# Matches the training run that produced grpo_v1
# Estimated time: ~90 min on T4, ~35 min on A100
import subprocess

result = subprocess.run(
    [
        "python", "train.py",
        "--backend", "trl",
        "--max-steps", "800",
        "--batch-size", "2",
        "--grad-accum", "8",
        "--learning-rate", "5e-6",
        "--output-dir", "./grpo_colab",
        "--task", "multi",
        "--logging-steps", "10",
        "--save-steps", "200",
    ],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Evaluate your freshly trained checkpoint
# Replace './grpo_colab' with './grpo_v1' to evaluate the pre-trained checkpoint
from inference import run_inference

checkpoint = "./grpo_colab"   # or "./grpo_v1" for the pre-trained model
tasks = ["weather_roulette", "dry_strategy_sprint", "late_safety_car", "championship_decider"]

print(f"Evaluating checkpoint: {checkpoint}")
print()
for task in tasks:
    try:
        scores = run_inference(checkpoint, task, n_episodes=2, verbose=False, seed=0)
        print(f"  {task:<32}  mean={sum(scores)/len(scores):.3f}  scores={[round(s,3) for s in scores]}")
    except Exception as e:
        print(f"  {task:<32}  ERROR: {e}")

## Summary

| What we showed | Evidence |
|---|---|
| Environment works end-to-end | Section 2 smoke test: reset → step → score |
| GRPO improves over untrained | Section 3 bar chart: +0.36 to +0.56 delta per scenario |
| Training actually happened | Section 4 curve: reward rises from 0.37 → 0.91 over 800 steps |
| Weather scenario is the key story | Section 5 race story: wrong tyre choice vs correct inter call |
| Generalizes across 8 circuits | Section 6 track grid: consistent expert-level performance |
| Training is reproducible | Section 7: anyone with a T4 can reproduce in ~90 min |

**Space:** https://huggingface.co/spaces/Deltasthic/f1-strategist  
**Blog:** https://huggingface.co/spaces/Deltasthic/f1-strategist/blob/main/blog.md  
**GitHub:** https://github.com/Deltasthicc/F1_Simulator_OpenENV